# Quill — Notebook 04: Entangled Knots

In this notebook, we will entangle **one qubit in Knot H with one qubit in Knot I**. Then we will measure the whole Cord and look for correlated characters.

**Goal:** see the difference between *superposition inside one register* and *entanglement between registers*.

We will use `Cord("HI")`: two character Knots, eight qubits per character, sixteen qubits total.

## 1. The idea: correlations that a product state cannot explain

A qubit can be in a superposition:

$$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

Two qubits prepared independently in this state have the product state:

$$|+\rangle\otimes|1\rangle = \frac{|01\rangle+|11\rangle}{\sqrt{2}}$$

But if we apply a controlled-NOT (CNOT) from the first qubit to the second, starting from $|+\rangle|0\rangle$, we get:

$$\mathrm{CNOT}\left(\frac{|00\rangle+|10\rangle}{\sqrt{2}}\right)
= \frac{|00\rangle+|11\rangle}{\sqrt{2}}$$

This is a **Bell state**. The outcomes are correlated: measuring the first qubit as 0 means the second is 0; measuring the first as 1 means the second is 1. Neither qubit alone has a definite value before measurement.

In Quill, we will make a similar pair across two character Knots. The full Cord will become a superposition of `HI` and `IH`.

## 2. Why `HI` is a useful example

ASCII values:

- `H` = 72 = binary `01001000`
- `I` = 73 = binary `01001001`

They differ only in the least-significant bit (bit 0). Our convention is **qubit 0 stores the least-significant bit**, even though binary strings are usually printed most-significant bit first.

We start from `Cord("HI")`. Then:

1. Apply Hadamard to bit 0 of the first Knot (`H`).
2. Use that qubit as the CNOT control and bit 0 of the second Knot (`I`) as the target.

Because the second Knot begins with bit 0 set to 1, the result is:

$$|HI\rangle \xrightarrow{H_{0}} \frac{|HI\rangle+|II\rangle}{\sqrt{2}}
\xrightarrow{\mathrm{CNOT}_{H_0\to I_0}} \frac{|HI\rangle+|IH\rangle}{\sqrt{2}}$$

Notice: the gate does not understand letters. It acts on encoded bits; the decoded characters change as a consequence.

## 3. Set up Qiskit

In [ ]:
%pip -q install qiskit qiskit-aer

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from collections import Counter

simulator = AerSimulator()

## 4. Build `Cord("HI")` and entangle the two Knots

We use 16 qubits: indices 0–7 for `H`, and 8–15 for `I`. The CNOT is between qubits 0 and 8. Classical bits are measured one-to-one with qubits.

In [ ]:
def cord_circuit(text):
    """Prepare one 8-qubit ASCII register per character, in LSB-first qubit order."""
    qc = QuantumCircuit(8 * len(text), 8 * len(text))
    for char_index, char in enumerate(text):
        value = ord(char)
        for bit_index in range(8):
            if (value >> bit_index) & 1:
                qc.x(8 * char_index + bit_index)
    return qc

qc = cord_circuit("HI")
qc.barrier(label="Cord HI")
qc.h(0)       # superposition on bit 0 of Knot H
qc.cx(0, 8)   # Knot H's bit 0 controls Knot I's bit 0
qc.barrier(label="entangled Knots")
qc.measure(range(16), range(16))

qc.draw("mpl")

## 5. Predict the state before measuring

The first Knot starts as `H`; the second starts as `I`. Only their bit-0 values can change in this circuit.

$$|HI\rangle = |0\rangle_{H_0}|1\rangle_{I_0}\otimes|\text{other bits}\rangle$$

After the Hadamard and CNOT, the relevant pair is:

$$\frac{|01\rangle+|10\rangle}{\sqrt{2}}$$

So the full Cord state is:

$$|\Psi\rangle = \frac{|HI\rangle+|IH\rangle}{\sqrt{2}}$$

Ideal measurement probabilities:

$$P(HI)=\frac12, \qquad P(IH)=\frac12$$

No other strings should appear in an ideal simulation.

## 6. Run many shots and decode the measured Cord

Qiskit prints classical bitstrings from the highest-index bit down to bit 0. The helper below reverses that display order, rebuilds each byte using LSB-first qubit indices, and decodes the ASCII characters.

In [ ]:
def decode_qiskit_key(key, n_chars=2):
    # Remove spaces between classical registers, if any, then restore c[0], c[1], ... order.
    bits_lsb_first = key.replace(" ", "")[::-1]
    chars = []
    for char_index in range(n_chars):
        byte_bits = bits_lsb_first[8*char_index:8*(char_index+1)]
        value = sum(int(bit) << bit_index for bit_index, bit in enumerate(byte_bits))
        chars.append(chr(value))
    return "".join(chars)

compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
string_counts = Counter()
for key, count in counts.items():
    string_counts[decode_qiskit_key(key)] += count

string_counts

You should see `HI` and `IH` at roughly 50% each. Small differences are ordinary sampling noise. The simulator is ideal, so this is not hardware noise.

In [ ]:
total = sum(string_counts.values())
for string, count in sorted(string_counts.items()):
    print(f"{string}: {count} shots ({count / total:.1%})")

## 7. What makes this entanglement?

The two relevant qubits are in the joint state:

$$|\Psi^-\rangle = \frac{|01\rangle+|10\rangle}{\sqrt{2}}$$

This state cannot be written as a product of a state for the first qubit and a state for the second. The measurement outcomes are linked: one is 0 exactly when the other is 1.

**Important distinction:** the whole 16-qubit Cord is entangled because of the gate we applied. Merely putting two Knots next to each other in a Cord does **not** entangle them. Also, the decoded strings are a way to inspect the encoded state—not proof of a quantum speedup.

## 8. Experiment

1. Change `qc.h(0)` to `qc.h(1)`, keeping the CNOT as written. What changes? Does the gate still create the intended entanglement between character bits?
2. Remove `qc.cx(0, 8)` and run again. What strings appear, and why?
3. Change the initial Cord to another pair of characters that differ in bit 0. Which pairs work with this exact circuit?

**Takeaway:** a gate can entangle qubits across Knots; measurement then reveals correlations between the decoded characters.